In [7]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = Path("Ultimate Guide To Coffee Beans.pdf")  # put PDF in same folder as notebook

print("Exists?", pdf_path.exists(), "| Location:", pdf_path.resolve())

loader = PyPDFLoader(str(pdf_path))
data = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(data)

print("Pages:", len(data))
print("Chunks:", len(chunks))


Exists? True | Location: C:\VS Code\SimpleRagCoffee\Ultimate Guide To Coffee Beans.pdf
Pages: 60
Chunks: 100


In [8]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate

embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = Chroma.from_documents(chunks, embedding=embeddings)
retriever = db.as_retriever(search_kwargs={"k": 1})

llm = ChatOllama(model="gemma3:4b", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context. If the answer isn't in the context, say you don't know."),
    ("human", "Question: {question}\n\nContext:\n{context}")
])

def ask(question: str):
    docs = retriever.invoke(question)  # NEW API
    context = "\n\n".join([f"[page {d.metadata.get('page', '?')}] {d.page_content}" for d in docs])
    answer = llm.invoke(prompt.format_messages(question=question, context=context)).content
    sources = [(d.metadata.get("source"), d.metadata.get("page", "?")) for d in docs]
    return answer, sources


In [9]:
answer, sources = ask("What are the main types of coffee beans?")
print(answer)
print("Sources:", sources)


The main types of coffee beans are Arabica, Robusta, Excelsa and Liberica.
Sources: [('Ultimate Guide To Coffee Beans.pdf', 0)]


In [10]:
answer, sources = ask("What is the best coffee bean?")
print(answer)
print("Sources:", sources)


Arabica coffee beans are the most widely consumed and highly regarded species of coffee bean. They’re known for their exquisite flavour profile.
Sources: [('Ultimate Guide To Coffee Beans.pdf', 0)]


In [11]:
answer, sources = ask("How many total types of coffee beans are there?")
print(answer)
print("Sources:", sources)


There are 4 main coffee bean types.
Sources: [('Ultimate Guide To Coffee Beans.pdf', 0)]


In [12]:
answer, sources = ask("Where can i get these coffee beans at?")
print(answer)
print("Sources:", sources)


I don't know.
Sources: [('Ultimate Guide To Coffee Beans.pdf', 2)]
